# 042 — Ingeniería y selección de características

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Transformaciones numéricas:** estandarizar `(x−μ)/σ` para modelos de distancia o
regularizados; log para colas largas; binning para no linealidades en lineales. Árboles:
invariantes a transformaciones monótonas.

**Categóricas:** one-hot (seguro, explota en cardinalidad), ordinal (solo con orden real),
target encoding (potente y peligroso: out-of-fold + suavizado
`(n·ȳ_c + k·ȳ)/(n+k)` obligatorios), hashing (colisiones controladas).

**Selección:** filtro (correlación/MI, barato, ciego a interacciones) → embedded
(lasso, importancias) → wrapper (RFE con CV anidada, caro).

**Regla anti-fuga:** todo parámetro aprendido (μ, σ, encodings, vocabularios, subconjunto
seleccionado) se ajusta SOLO con train, dentro del pipeline que se valida. Seleccionar
features con el dataset completo y "luego validar" infla la métrica (ESL §7.10.2).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Target encoding con suavizado.** Media global de impago ȳ = 0.08, k = 20.
Calcula el encoding suavizado `(n·ȳ_c + k·ȳ)/(n + k)` para: ciudad X (n=200, ȳ_c=0.15),
ciudad Y (n=15, ȳ_c=0.40) y ciudad Z (n=1, ȳ_c=1.00). ¿Cuál se aleja más de su media
observada y por qué eso es deseable?

**Ejercicio 2 — Detectar la fuga de selección.** Con 5 000 features de ruido puro y
n = 100 ejemplos binarios balanceados, un colega selecciona las 10 features más
correlacionadas con el target usando TODO el dataset y luego reporta CV accuracy 0.78.
Explica el mecanismo exacto de la inflación y qué resultado esperarías con el protocolo
correcto (selección dentro de cada fold).

**Ejercicio 3 — Codificación cíclica.** La hora del día (0-23) se codifica como
`sin(2πh/24), cos(2πh/24)`. Calcula las coordenadas para h = 23 y h = 1 y su distancia
euclidiana; compárala con la distancia si usaras la hora cruda (|23−1| = 22). ¿Qué modelo
se beneficia y a cuál le da igual?

**Ejercicio 4 — Invariancia en el laboratorio.** Ejecuta `run_lab("ml", seed=42)` (celda
TODO). Si aplicaras `log(x)` a la feature antes del barrido de umbrales, ¿cambiaría la
accuracy alcanzable? ¿Y si aplicaras `−x`? Razona con monotonía antes de responder.


In [ ]:
# TODO: ejecuta run_lab("ml", seed=42)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: target encoding suavizado
y_global, k = 0.08, 20

def encoding(n_c, y_c):
    return None  # completa: (n_c·y_c + k·y_global) / (n_c + k)

for nombre, n_c, y_c in [("X", 200, 0.15), ("Y", 15, 0.40), ("Z", 1, 1.00)]:
    print(nombre, encoding(n_c, y_c))


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 3: codificación cíclica de la hora
import math

def cic(h):
    return None  # completa: (sin(2πh/24), cos(2πh/24))

# distancia euclidiana entre h=23 y h=1 en el círculo vs |23−1| en crudo


## Reflexión

1. El laboratorio trabaja con una única feature sintética. Si le añadieras 99 features de
   ruido puro y seleccionaras "la mejor" por accuracy antes de validar, ¿qué accuracy de
   validación esperarías para la ganadora y por qué está inflada?
2. ¿Por qué el target encoding sin out-of-fold puede dar validación excelente y producción
   desastrosa, mientras el one-hot no tiene ese modo de fallo?
3. ¿Qué transformación de esta clase cambiaría el resultado del laboratorio (que elige un
   umbral sobre una feature) y cuál lo dejaría exactamente igual? Justifica con la
   invariancia monótona.
